In [1]:
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)

def sim_Data(T, N, m, p, f, with_covariates):
    Outcomes = []
    Thetas = []
    Lambdas = []
    latent_terms = []

    # Fixed base loadings
    Theta_base = torch.randn(m, p, dtype=torch.float64)
    Lambda_base = torch.randn(m, f, dtype=torch.float64)

    # step directions
    Theta_step = torch.randn(m, p, dtype=torch.float64)
    Lambda_step = torch.randn(m, f, dtype=torch.float64)

    for t in range(T):
        if t == 0:
            # start out at the base, before taking any steps
            Theta_t = Theta_base
            Lambda_t = Lambda_base
        else:
            # take a step, always in the same direction, but by varying amounts
            Theta_t = Theta_base + (Theta_step * torch.abs(torch.randn(1)))
            Lambda_t = Lambda_base + (Lambda_step * torch.abs(torch.randn(1)))
        # each time point append
        Thetas.append(Theta_t)
        Lambdas.append(Lambda_t)

    # if we aren't using covariates, just do the loadings for the latent factors
    Z_F = torch.randn(p, N, dtype=torch.float64) * bool(with_covariates)  # (p x N)
    M_F= torch.randn(f, N, dtype=torch.float64)  # (f x N)

    for t in range(T):
        # returning the latent terms
        L_t = Lambdas[t] @ M_F
        latent_terms.append(L_t)

        Y_t = Thetas[t] @ Z_F + Lambdas[t] @ M_F
        Outcomes.append(Y_t)

    return Outcomes, latent_terms, M_F, Z_F




## Differentiating along orthogonal matrices does give a different final Q matrix, but idential predictions!

In [2]:


import torch
import numpy as np
# torch.manual_seed(42)
# np.random.seed(42)
from learnQorthogonal import learnQorthogonal
embedding_dim = 5
# settings
T = 10
N = 20
m = 5
f = 1
p = 1

with_covariates = True

Outcomes, latent_terms, M_F,_ = sim_Data(T, N, m, p, f, with_covariates)

targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# using this orthogonally constrained, fixed weights version is: 
# faster, more natural (no regularization), more interpretable (simple average of synthetic donors), and perhaps more mathematically tractable.
# Also, when the number of units and treatment periods is large, it fits extremely well.
# in addition, the norm of the covariates is roughly equal to the norm of the synthetic covariates (if the embedding dimension is close to N).
Q_final, w_final = learnQorthogonal(targets, covariates, embedding_dim=embedding_dim, n_iterations=1000,
                           reg_Q=0, reg_w=0, verbose=False, num_timepoints = T, init_Q = "eye", fixed_weights=True)

print("type of Q:", type(Q_final))
print("type of w:", type(w_final))
print("################################")
#print("Q_final: \n", Q_final)
print("w_final: \n", w_final)
print("Q_final: \n", Q_final)
print("Q @ w: \n", torch.from_numpy(Q_final) @ w_final)
print("Synthetic Covariates: \n", covariates[-1] @ Q_final)

U,S,Vh = np.linalg.svd(covariates[-1], full_matrices=False)
print("SVD of covariates: \n", )
print("U: ",U)
print("S: ",S)
print("Vh: ",Vh)

print("Ground Truth at treatment: ", targets[-1])
pred_Q = covariates[-1] @ Q_final @ w_final
print("Prediction at treatment: ", pred_Q)
bias = (pred_Q - targets[-1])
print("Bias at treatment time: ", torch.norm(bias))

(CVXPY) Apr 26 12:57:35 PM: Encountered unexpected exception importing solver DIFFCP:
ImportError('diffcp >= 1.0.15 is required')
type of Q: <class 'numpy.ndarray'>
type of w: <class 'numpy.ndarray'>
################################
w_final: 
 [0.2 0.2 0.2 0.2 0.2]
Q_final: 
 [[ 0.86478985 -0.16521748 -0.14957736 -0.17097606 -0.22934123]
 [-0.0552752   0.90111337 -0.07648459 -0.10665224 -0.15801232]
 [-0.09610774 -0.13259161  0.88626485 -0.1393006  -0.19428787]
 [-0.04150976 -0.0877816  -0.06405219  0.90405126 -0.14604142]
 [-0.00321987 -0.06881551 -0.0350943  -0.08050545  0.86144666]
 [ 0.12446036  0.08945009  0.10727535  0.08355144  0.08112842]
 [ 0.07655676  0.05032829  0.06379346  0.04572529  0.03879877]
 [ 0.15600444  0.11527087  0.13593458  0.1085319   0.10907535]
 [ 0.07349549  0.04784107  0.06102091  0.04332325  0.03610751]
 [ 0.13169391  0.09537419  0.11384901  0.08928315  0.08753899]
 [-0.0504139  -0.0494007  -0.04956394 -0.04982115 -0.06846999]
 [ 0.13169783  0.0953774   0.1

In [87]:
synthetic_covariates = covariates[-1] @ Q_final

print("Shape of covariates: ", covariates[-1].shape)
print("norm of covariates[-1]: ", np.linalg.norm(covariates[-1]))

print("Shape of synthetic covariates: ", synthetic_covariates.shape)
print("norm of synthetic covariates: ", np.linalg.norm(covariates[-1] @ Q_final))

print("mean of covariates[-1]: ", torch.mean(covariates[-1]))
print("mean of synthetic covariates: ", torch.mean(synthetic_covariates))

Shape of covariates:  torch.Size([5, 19])
norm of covariates[-1]:  18.822713533427127
Shape of synthetic covariates:  torch.Size([5, 5])
norm of synthetic covariates:  8.777216026523467
mean of covariates[-1]:  tensor(-0.01, dtype=torch.float64)
mean of synthetic covariates:  tensor(0.26, dtype=torch.float64)


# Claude wrote this. Thanks Claude

In [86]:
# Construct A and b from existing variables
A = np.vstack([c.numpy() for c in covariates])  # shape (T*m, N)
b = np.concatenate([t.numpy() for t in targets])  # shape (T*m,)

r = 1.0 / np.sqrt(embedding_dim)

# --- Secular equation (exact global optimum on sphere) ---
U_a, S_a, Vt_a = np.linalg.svd(A, full_matrices=False)
UTb = U_a.T @ b

def gamma_of_lambda(lam):
    return Vt_a.T @ ((S_a / (S_a**2 + lam)) * UTb)

def norm_residual(lam):
    return np.linalg.norm(gamma_of_lambda(lam)) - r

from scipy.optimize import brentq

# Check if unconstrained solution is already inside the ball
gamma_unconstrained = gamma_of_lambda(0)
print("Unconstrained ‖γ‖:", np.linalg.norm(gamma_unconstrained).round(6))
print("Target radius r:   ", round(r, 6))

if np.linalg.norm(gamma_unconstrained) <= r:
    print("Unconstrained optimum is INSIDE the ball — constraint will be active from outside")
    lam_star = brentq(norm_residual, -S_a[-1]**2 + 1e-10, 0)  # lambda can be negative
else:
    lam_star = brentq(norm_residual, 0, 1e8)

gamma_exact = gamma_of_lambda(lam_star)

# --- Compare all three ---
gamma_Q = Q_final @ w_final.numpy() if isinstance(w_final, torch.Tensor) else Q_final @ w_final

print("\n--- Norms ---")
print("‖γ_exact‖:         ", np.linalg.norm(gamma_exact).round(6))
print("‖γ_Q‖:             ", np.linalg.norm(gamma_Q).round(6))
print("‖γ_unconstrained‖: ", np.linalg.norm(gamma_unconstrained).round(6))

print("\n--- Objectives ---")
print("Exact sphere:   ", np.sum((A @ gamma_exact - b)**2).round(6))
print("Q method:       ", np.sum((A @ gamma_Q - b)**2).round(6))
print("Unconstrained:  ", np.sum((A @ gamma_unconstrained - b)**2).round(6))

print("\n--- γ vectors ---")
print("γ_exact: ", gamma_exact.round(4))
print("γ_Q:     ", gamma_Q.round(4))

Unconstrained ‖γ‖: 1.538868
Target radius r:    0.447214

--- Norms ---
‖γ_exact‖:          0.173116
‖γ_Q‖:              0.447214
‖γ_unconstrained‖:  1.538868

--- Objectives ---
Exact sphere:    0.0
Q method:        0.0
Unconstrained:   0.0

--- γ vectors ---
γ_exact:  [ 0.0307 -0.0547  0.0296  0.0405  0.0006  0.0282  0.0433  0.083  -0.0062
 -0.0363 -0.0473  0.0471 -0.011  -0.0043 -0.0481 -0.0691  0.0299  0.0019
 -0.0049]
γ_Q:      [ 0.2055  0.1562  0.1977  0.1956  0.1866  0.0299  0.0352  0.0401 -0.0016
 -0.0438 -0.0332  0.0533 -0.0391 -0.0219 -0.0316 -0.0412  0.0581  0.0186
 -0.0562]


Shape of covariates:  torch.Size([3, 4])
norm of covariates[-1]:  3.7823820072245486
Shape of synthetic covariates:  torch.Size([3, 5])
norm of synthetic covariates:  3.782382007224549
mean of covariates[-1]:  tensor(-0.44, dtype=torch.float64)
mean of synthetic covariates:  tensor(0.46, dtype=torch.float64)


In [5]:
torch.manual_seed(42)
np.random.seed(42)

from learnQ import learnQ

# settings
T = 40
N = 50
m = 10
f = 3
# since we aren't passing covariates, p doesn't matter here.
p = 2

with_covariates = False

Outcomes, latent_terms, M_F,_ = sim_Data(T, N, m, p, f, with_covariates)

targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])


Q_final, w_final = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, num_timepoints = None, init_Q = "eye")

print("Q_final shape:", Q_final.shape)
print("w_final shape:", w_final.shape)
print("Q_final: \n", Q_final)
print("synthetic covariates: \n", covariates[-1] @ Q_final @ w_final)


Q_final shape: (49, 3)
w_final shape: (3,)
Q_final: 
 [[ 0.86653042  0.0020883  -0.07371828]
 [-0.12804315  0.29004644 -0.08004263]
 [-0.12513638 -0.036888    0.80687887]
 [ 0.10215684  0.00823739  0.08678396]
 [-0.15795274 -0.04910802  0.01104044]
 [ 0.11503393  0.00578723  0.04698654]
 [-0.14596763 -0.06393904 -0.03006375]
 [ 0.14044282  0.02020672  0.02364619]
 [-0.12299091 -0.03206301 -0.06575211]
 [-0.10587769  0.00521766 -0.05739446]
 [-0.01740133 -0.05523227  0.04970917]
 [-0.09847208  0.01286592 -0.07784064]
 [-0.123835   -0.03209605 -0.06343159]
 [-0.08325526  0.0209444  -0.06994565]
 [ 0.1245723   0.04042612  0.0688092 ]
 [-0.11216629 -0.01033668 -0.0802384 ]
 [-0.11653488 -0.01607584 -0.06685041]
 [ 0.12045054  0.01516623  0.06058347]
 [ 0.12002233  0.02737206  0.07562014]
 [-0.03144583  0.0513547  -0.06131812]
 [ 0.17843367  0.0699488   0.07833763]
 [ 0.10226871 -0.00558432  0.06083524]
 [ 0.124721    0.04304958  0.07198191]
 [-0.1257311  -0.0362855  -0.05922726]
 [-0.06281

In [ ]:
print(Q_final)
print(Q_final[:,0:1])
# norm of columns is 1
print(np.linalg.norm(Q_final[:,0:1]))
first_col = Q_final[:,0:1]
second_col = Q_final[:,1:2]

# shows cols are orthogonal
print(np.dot(first_col.T, second_col))



Need to see what Monte Carlo Bias looks like

In [ ]:
pred_Q = covariates[-1] @ Q_final @ w_final
print(pred_Q)
bias = (pred_Q - targets[-1])
print(torch.norm(bias))

nMinus1, D = Q_final.shape
minDim = min(nMinus1, D)
print(np.linalg.norm(Q_final.T @ Q_final - np.eye(minDim)))

## Initializing with different matrices seems to give same solution for prediction

In [ ]:
# Run with two different initializations
Q1, w1 = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, init_Q = "id")
Q2, w2 = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, init_Q = "random")

# do predictions agree?
pred1 = covariates[-1] @ Q1 @ w1
pred2 = covariates[-1] @ Q2 @ w2

In [ ]:
print(pred1)
print(pred2)